In [20]:
import numpy as np

In [2]:
import pandas as pd

df = pd.read_csv("../data/web-page-phishing.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100077, 20)


,url_length,n_dots,n_hypens,n_underline,n_slash,n_questionmark,n_equal,n_at,n_and,n_exclamation,n_space,n_tilde,n_comma,n_plus,n_asterisk,n_hastag,n_dollar,n_percent,n_redirection,phishing
0,37,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,77,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
2,126,4,1,2,0,1,3,0,2,0,0,0,0,0,0,0,0,0,1,1
3,18,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,55,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


In [3]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['url_length', 'n_dots', 'n_hypens', 'n_underline', 'n_slash', 'n_questionmark', 'n_equal', 'n_at', 'n_and', 'n_exclamation', 'n_space', 'n_tilde', 'n_comma', 'n_plus', 'n_asterisk', 'n_hastag', 'n_dollar', 'n_percent', 'n_redirection', 'phishing']

Data types:
url_length        int64
n_dots            int64
n_hypens          int64
n_underline       int64
n_slash           int64
n_questionmark    int64
n_equal           int64
n_at              int64
n_and             int64
n_exclamation     int64
n_space           int64
n_tilde           int64
n_comma           int64
n_plus            int64
n_asterisk        int64
n_hastag          int64
n_dollar          int64
n_percent         int64
n_redirection     int64
phishing          int64
dtype: object

Missing values:
0

Duplicate rows:
78186


In [4]:
print("Phishing distribution:")
print(df["phishing"].value_counts())

print("\nPhishing distribution (%):")
print(df["phishing"].value_counts(normalize=True) * 100)

Phishing distribution:
phishing
0    63715
1    36362
Name: count, dtype: int64

Phishing distribution (%):
phishing
0    63.665977
1    36.334023
Name: proportion, dtype: float64


In [5]:
feature_columns = [col for col in df.columns if col != "phishing"]

duplicate_groups = df.groupby(feature_columns)["phishing"].nunique()

print("Duplicate feature patterns with conflicting labels:",
      (duplicate_groups > 1).sum())

Duplicate feature patterns with conflicting labels: 1913


In [6]:
conflicting_patterns = duplicate_groups[duplicate_groups > 1]

print("Number of conflicting feature patterns:", len(conflicting_patterns))
print("Rows involved in conflicting patterns:",
      df.groupby(feature_columns)["phishing"]
        .transform("nunique")
        .gt(1)
        .sum())

Number of conflicting feature patterns: 1913
Rows involved in conflicting patterns: 61586


In [7]:
print("Unique feature patterns:", len(duplicate_groups))
print("Duplicate rows:", df.duplicated().sum())

Unique feature patterns: 19978
Duplicate rows: 78186


In [8]:
conflict_rows = df[
    df.groupby(feature_columns)["phishing"].transform("nunique") > 1
]

print(conflict_rows["phishing"].value_counts())
print(conflict_rows.head(20))

phishing
0    48423
1    13163
Name: count, dtype: int64
    url_length  n_dots  n_hypens  n_underline  n_slash  n_questionmark  \
0           37       3         0            0        0               0   
3           18       2         0            0        0               0   
5           32       3         1            0        0               0   
6           19       2         0            0        0               0   
8           42       2         0            0        0               0   
13          31       2         0            0        0               0   
14          25       2         0            0        0               0   
15          51       3         0            0        0               1   
16          31       2         0            0        0               0   
17          50       2         0            0        0               0   
18          34       2         0            0        0               0   
19          27       2         0            0        0 

In [9]:
pattern_stats = (
    df.groupby(feature_columns)["phishing"]
    .agg(["count", "mean"])
    .reset_index()
)

pattern_stats["majority_label"] = (pattern_stats["mean"] >= 0.5).astype(int)

print("Total unique feature patterns:", len(pattern_stats))

print("\nPattern frequency:")
print(pattern_stats["count"].describe())

print("\nPatterns with conflicting labels:")
print(((pattern_stats["mean"] > 0) & (pattern_stats["mean"] < 1)).sum())

print("\nRows represented by conflicting patterns:")
print(
    pattern_stats.loc[
        (pattern_stats["mean"] > 0) & (pattern_stats["mean"] < 1),
        "count"
    ].sum()
)

Total unique feature patterns: 19978

Pattern frequency:
count    19978.000000
mean         5.009360
std         38.747094
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max       1540.000000
Name: count, dtype: float64

Patterns with conflicting labels:
1913

Rows represented by conflicting patterns:
61586


In [10]:
feature_summary = df.describe().T

print(feature_summary[["min", "max", "mean", "std"]])

                min     max       mean        std
url_length      4.0  4165.0  39.177683  47.971847
n_dots          1.0    24.0   2.224397   1.255046
n_hypens        0.0    43.0   0.405188   1.285465
n_underline     0.0    21.0   0.137724   0.723995
n_slash         0.0    44.0   1.135386   1.828526
n_questionmark  0.0     9.0   0.024391   0.167789
n_equal         0.0    23.0   0.215834   0.959802
n_at            0.0    43.0   0.022143   0.268393
n_and           0.0    26.0   0.143330   0.913656
n_exclamation   0.0    10.0   0.002608   0.082207
n_space         0.0    18.0   0.004876   0.144569
n_tilde         0.0     5.0   0.003617   0.078500
n_comma         0.0    11.0   0.002378   0.079558
n_plus          0.0    19.0   0.002468   0.104382
n_asterisk      0.0    60.0   0.004097   0.284046
n_hastag        0.0    13.0   0.000450   0.058028
n_dollar        0.0    10.0   0.001899   0.097412
n_percent       0.0   174.0   0.109286   1.695327
n_redirection  -1.0    17.0   0.361532   0.775492


In [11]:
feature_columns = [col for col in df.columns if col != "phishing"]

pattern_stats = (
    df.groupby(feature_columns)["phishing"]
    .agg(["count", "mean", "nunique"])
    .reset_index()
)

print("Unique feature patterns:", len(pattern_stats))
print("Conflicting patterns:", (pattern_stats["nunique"] > 1).sum())

conflicting_patterns = pattern_stats[pattern_stats["nunique"] > 1]

print("\nConflicting pattern rows:")
print(conflicting_patterns["count"].sum())

print("\nConflicting pattern label distribution:")
print(
    df[df.groupby(feature_columns)["phishing"].transform("nunique") > 1]
    ["phishing"]
    .value_counts()
)

Unique feature patterns: 19978
Conflicting patterns: 1913

Conflicting pattern rows:
61586

Conflicting pattern label distribution:
phishing
0    48423
1    13163
Name: count, dtype: int64


In [12]:
duplicate_counts = (
    df.groupby(feature_columns)
    .size()
    .reset_index(name="count")
)

print("Total rows:", len(df))
print("Unique feature patterns:", len(duplicate_counts))

print("\nPatterns appearing once:")
print((duplicate_counts["count"] == 1).sum())

print("\nPatterns appearing more than once:")
print((duplicate_counts["count"] > 1).sum())

print("\nRows belonging to repeated patterns:")
print(df.groupby(feature_columns).size().loc[lambda x: x > 1].sum())

Total rows: 100077
Unique feature patterns: 19978

Patterns appearing once:
13966

Patterns appearing more than once:
6012

Rows belonging to repeated patterns:
86111


In [13]:
clean_df = df.drop_duplicates()

print("Original rows:", len(df))
print("Rows after exact duplicate removal:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

print("\nClass distribution after duplicate removal:")
print(clean_df["phishing"].value_counts())

print("\nClass distribution (%):")
print(clean_df["phishing"].value_counts(normalize=True) * 100)

Original rows: 100077
Rows after exact duplicate removal: 21891
Rows removed: 78186

Class distribution after duplicate removal:
phishing
1    15872
0     6019
Name: count, dtype: int64

Class distribution (%):
phishing
1    72.504682
0    27.495318
Name: proportion, dtype: float64


In [14]:
feature_columns = [col for col in clean_df.columns if col != "phishing"]

pattern_stats = (
    clean_df.groupby(feature_columns)["phishing"]
    .agg(["count", "mean"])
    .reset_index()
)

pattern_stats["phishing"] = (pattern_stats["mean"] >= 0.5).astype(int)

processed_df = pattern_stats.drop(columns=["mean"])

print("Rows after resolving conflicting patterns:", len(processed_df))

print("\nClass distribution:")
print(processed_df["phishing"].value_counts())

print("\nClass distribution (%):")
print(processed_df["phishing"].value_counts(normalize=True) * 100)

Rows after resolving conflicting patterns: 19978

Class distribution:
phishing
1    15872
0     4106
Name: count, dtype: int64

Class distribution (%):
phishing
1    79.447392
0    20.552608
Name: proportion, dtype: float64


In [15]:
print("Final dataset shape:", processed_df.shape)
print("\nDuplicate rows:", processed_df.duplicated().sum())

feature_columns = [col for col in processed_df.columns if col != "phishing"]

print("Duplicate feature patterns:", processed_df.duplicated(subset=feature_columns).sum())

print("\nClass distribution:")
print(processed_df["phishing"].value_counts())

print("\nClass distribution (%):")
print(processed_df["phishing"].value_counts(normalize=True) * 100)

Final dataset shape: (19978, 21)

Duplicate rows: 0
Duplicate feature patterns: 0

Class distribution:
phishing
1    15872
0     4106
Name: count, dtype: int64

Class distribution (%):
phishing
1    79.447392
0    20.552608
Name: proportion, dtype: float64


In [16]:
X = processed_df.drop(columns=["phishing"])
y = processed_df["phishing"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (19978, 20)
y shape: (19978,)


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)
print("\nTraining distribution:")
print(y_train.value_counts(normalize=True) * 100)
print("\nValidation distribution:")
print(y_val.value_counts(normalize=True) * 100)
print("\nTest distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training set: (13984, 20)
Validation set: (2997, 20)
Test set: (2997, 20)

Training distribution:
phishing
1    79.447941
0    20.552059
Name: proportion, dtype: float64

Validation distribution:
phishing
1    79.446113
0    20.553887
Name: proportion, dtype: float64

Test distribution:
phishing
1    79.446113
0    20.553887
Name: proportion, dtype: float64


In [21]:
print("Missing values:")
print(processed_df.isnull().sum())

print("\nInfinite values:")
print(np.isinf(processed_df.select_dtypes(include=np.number)).sum())

print("\nUnique values per feature:")
print(processed_df.nunique().sort_values())

print("\nConstant features:")
constant_features = [
    col for col in X_train.columns
    if X_train[col].nunique() <= 1
]

print(constant_features)

Missing values:
url_length        0
n_dots            0
n_hypens          0
n_underline       0
n_slash           0
n_questionmark    0
n_equal           0
n_at              0
n_and             0
n_exclamation     0
n_space           0
n_tilde           0
n_comma           0
n_plus            0
n_asterisk        0
n_hastag          0
n_dollar          0
n_percent         0
n_redirection     0
count             0
phishing          0
dtype: int64

Infinite values:
url_length        0
n_dots            0
n_hypens          0
n_underline       0
n_slash           0
n_questionmark    0
n_equal           0
n_at              0
n_and             0
n_exclamation     0
n_space           0
n_tilde           0
n_comma           0
n_plus            0
n_asterisk        0
n_hastag          0
n_dollar          0
n_percent         0
n_redirection     0
count             0
phishing          0
dtype: int64

Unique values per feature:
phishing            2
count               2
n_questionmark      6
n_tild

In [22]:
print("Training feature statistics:")
print(X_train.describe().T)

Training feature statistics:
                  count       mean        std  min   25%   50%   75%     max
url_length      13984.0  82.706379  82.576875  4.0  44.0  64.0  92.0  4165.0
n_dots          13984.0   2.945867   2.186211  1.0   2.0   2.0   3.0    24.0
n_hypens        13984.0   1.385512   2.457719  0.0   0.0   1.0   2.0    43.0
n_underline     13984.0   0.523741   1.438782  0.0   0.0   0.0   0.0    21.0
n_slash         13984.0   2.688287   2.499549  0.0   0.0   2.0   4.0    44.0
n_questionmark  13984.0   0.088601   0.305288  0.0   0.0   0.0   0.0     9.0
n_equal         13984.0   0.779391   1.795375  0.0   0.0   0.0   1.0    20.0
n_at            13984.0   0.093893   0.650654  0.0   0.0   0.0   0.0    43.0
n_and           13984.0   0.539474   1.820293  0.0   0.0   0.0   0.0    26.0
n_exclamation   13984.0   0.010655   0.176462  0.0   0.0   0.0   0.0     9.0
n_space         13984.0   0.023026   0.337058  0.0   0.0   0.0   0.0    18.0
n_tilde         13984.0   0.014803   0.164403  

In [23]:
correlation_matrix = X_train.corr()

print(correlation_matrix.round(2))

                url_length  n_dots  n_hypens  n_underline  n_slash  \
url_length            1.00    0.38      0.27         0.32     0.21   
n_dots                0.38    1.00     -0.03         0.07     0.09   
n_hypens              0.27   -0.03      1.00         0.07    -0.01   
n_underline           0.32    0.07      0.07         1.00     0.06   
n_slash               0.21    0.09     -0.01         0.06     1.00   
n_questionmark        0.29    0.34     -0.01         0.09    -0.11   
n_equal               0.54    0.51      0.03         0.25     0.10   
n_at                  0.12    0.12     -0.02         0.02     0.03   
n_and                 0.51    0.58      0.02         0.20     0.07   
n_exclamation         0.04   -0.01     -0.01         0.05     0.01   
n_space               0.04   -0.00      0.01        -0.01    -0.05   
n_tilde               0.10   -0.00     -0.02         0.02     0.04   
n_comma               0.05    0.03      0.00         0.02     0.05   
n_plus              

In [24]:
train_df = X_train.copy()
train_df["phishing"] = y_train.values

val_df = X_val.copy()
val_df["phishing"] = y_val.values

test_df = X_test.copy()
test_df["phishing"] = y_test.values

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("Datasets saved successfully.")

Datasets saved successfully.


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled,
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled training shape: (13984, 20)
Scaled validation shape: (2997, 20)
Scaled test shape: (2997, 20)


In [26]:
print("Training means:")
print(X_train_scaled.mean().round(4))

print("\nTraining standard deviations:")
print(X_train_scaled.std().round(4))

Training means:
url_length       -0.0
n_dots            0.0
n_hypens          0.0
n_underline      -0.0
n_slash          -0.0
n_questionmark   -0.0
n_equal           0.0
n_at             -0.0
n_and             0.0
n_exclamation    -0.0
n_space          -0.0
n_tilde           0.0
n_comma          -0.0
n_plus           -0.0
n_asterisk        0.0
n_hastag         -0.0
n_dollar         -0.0
n_percent        -0.0
n_redirection     0.0
count             0.0
dtype: float64

Training standard deviations:
url_length        1.0
n_dots            1.0
n_hypens          1.0
n_underline       1.0
n_slash           1.0
n_questionmark    1.0
n_equal           1.0
n_at              1.0
n_and             1.0
n_exclamation     1.0
n_space           1.0
n_tilde           1.0
n_comma           1.0
n_plus            1.0
n_asterisk        1.0
n_hastag          1.0
n_dollar          1.0
n_percent         1.0
n_redirection     1.0
count             1.0
dtype: float64


In [27]:
train_scaled = X_train_scaled.copy()
train_scaled["phishing"] = y_train.values

validation_scaled = X_val_scaled.copy()
validation_scaled["phishing"] = y_val.values

test_scaled = X_test_scaled.copy()
test_scaled["phishing"] = y_test.values

train_scaled.to_csv("train_scaled.csv", index=False)
validation_scaled.to_csv("validation_scaled.csv", index=False)
test_scaled.to_csv("test_scaled.csv", index=False)

print("Scaled datasets saved successfully.")

Scaled datasets saved successfully.


In [28]:
import joblib

joblib.dump(scaler, "../models/scaler.pkl")

print("Scaler saved successfully.")

Scaler saved successfully.
